# 🧠 CalRetail — First Party Audience Segmentation
## Goal
Cluster shoppers into distinct, well-labelled cohorts using the full real RFM + engagement
feature set.

## Algorithmic Explanation
**Standardized K-Means Clustering**
1. Use the pre-engineered customer feature table (`feature_customers.csv`): real recency,
   frequency, monetary, average order value, browsing, wishlist and cart-add counts — not just a
   thin recency/frequency/monetary subset recomputed from raw transactions.
2. Scale numerical metrics using StandardScaler.
3. Pick the number of clusters via the elbow method on real inertia
   (`adaptive_thresholds.get_kmeans_optimal_k`) by default, instead of a fixed k=4.
4. Label each cluster from its real centroid characteristics
   (`adaptive_thresholds.get_auto_cluster_labels`) with deduplicated, business-friendly names.



In [ ]:
import os
import sys
import numpy as np
import pandas as pd
from pathlib import Path
import warnings
import json
import re
import math
warnings.filterwarnings('ignore')

# Set path to include parent directory
base_path = Path().resolve()
while not (base_path / 'data').exists() and base_path.parent != base_path:
    base_path = base_path.parent
processed_dir = base_path / 'data' / 'processed'
if str(base_path) not in sys.path:
    sys.path.insert(0, str(base_path))

print(f"Project root found at: {base_path}")
print(f"Data directory: {processed_dir}")

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from backend.utils.adaptive_thresholds import get_kmeans_optimal_k, get_auto_cluster_labels

# Use the full, pre-engineered RFM + engagement feature table instead of
# recomputing a thin recency/frequency/monetary subset from raw transactions.
rfm = pd.read_csv(processed_dir / 'feature_customers.csv')

FEATURE_COLS = [c for c in [
    'recency_days', 'frequency', 'monetary', 'avg_order_val',
    'total_browses', 'wishlist_count', 'cart_count'
] if c in rfm.columns]

scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm[FEATURE_COLS].fillna(0))

# Elbow-method k, derived from real inertia curve on this data, instead of a
# fixed k=4 for every request.
DEFAULT_K = get_kmeans_optimal_k()
kmeans = KMeans(n_clusters=DEFAULT_K, random_state=42, n_init=10)
rfm['cluster'] = kmeans.fit_predict(rfm_scaled)

print(f"Applied K-Means with data-derived k={DEFAULT_K} (elbow method). Generated {rfm['cluster'].nunique()} user clusters.")

In [ ]:
def list_audience_segments(n_clusters=None):
    global rfm, rfm_scaled

    k = int(n_clusters) if n_clusters else DEFAULT_K
    kmeans_dyn = KMeans(n_clusters=k, random_state=42, n_init=10)
    rfm['cluster'] = kmeans_dyn.fit_predict(rfm_scaled)

    cluster_summaries = []
    for cluster_id in sorted(rfm['cluster'].unique()):
        cd = rfm[rfm['cluster'] == cluster_id]
        cluster_summaries.append({
            "cluster_id": int(cluster_id),
            "size": int(len(cd)),
            "avg_recency_days": float(cd['recency_days'].mean()) if 'recency_days' in cd.columns else 30.0,
            "avg_frequency": float(cd['frequency'].mean()) if 'frequency' in cd.columns else 5.0,
            "avg_monetary": float(cd['monetary'].mean()) if 'monetary' in cd.columns else 1000.0,
            "avg_browse_count": float(cd['total_browses'].mean()) if 'total_browses' in cd.columns else 10.0,
        })

    # Real centroid-based labels (recency/frequency/monetary/browse relative
    # to the population medians), deduplicated — replaces a simpler nested-if
    # scheme that could assign the same label to multiple clusters.
    labels = get_auto_cluster_labels(cluster_summaries)

    results = []
    for s in cluster_summaries:
        results.append({
            "cluster_id": s["cluster_id"],
            "size": s["size"],
            "label": labels.get(s["cluster_id"], "Occasional Buyers"),
            "avg_recency": round(s["avg_recency_days"], 1),
            "avg_frequency": max(1, int(round(s["avg_frequency"]))),
            "avg_spend": round(s["avg_monetary"], 2),
            "avg_browse_count": round(s["avg_browse_count"], 1),
        })
    return results

segments = list_audience_segments()
print("Audience Cohorts payload:\n", json.dumps(segments, indent=2))

In [ ]:
print("=== CALRETAIL COHORTS SEGMENTATION MAP ===")
seg_df = pd.DataFrame(segments)
print(seg_df[['cluster_id', 'label', 'size', 'avg_recency', 'avg_spend']].to_string(index=False))
